# 02 · Silver · `inspecoes`
### V-Energy Grid 360 — Arquitetura Medalhão

Lê `v_energy.bronze.inspecoes` (as 3 abas de `inspecoes.xlsx`, unidas na Bronze com a coluna `aba_origem`) e grava `v_energy.silver.inspecoes`. **Grão: 1 linha = 1 inspeção de campo.**

**Fonte das regras:** `exploracao_inspecoes.ipynb` e a seção `inspecoes` do `docs/data_quality_log.md` (I1 a I5).

| # | Tratamento | Linhas afetadas | Regra | Log |
|---|---|---|---|---|
| 1 | `data` em 4 formatos | 133 fora do ISO | cada padrão com o seu formato (hífen = mês primeiro) | I1 |
| 2 | Data fora do período da aba | 107 (todas na aba `dez`) | manter a data + flag `data_fora_da_aba` | I2 |
| 3 | `resultado` com 8 grafias | 164 fora do padrão | **mapa explícito** para 2 valores + flag `irregularidade_confirmada` | I3 |
| 4 | `tipo_irregularidade` ausente em 2 formas (nulo e `-`) | 518 | nulo real | I4 |
| 5 | `kwh_recuperado` com vírgula decimal | 125 | vírgula vira ponto, `decimal(12,2)` | I5 |

**Nenhuma linha é removida:** a tabela não tem cópias, órfãs nem UCs de teste (exploração, seções 3, 5 e 10).

**Por que o cuidado com o resultado:** é a coluna que diz se a inspeção encontrou irregularidade, e é ela que calibra a regra da `fila_inspecao`. Por isso o mapa é **valor a valor**: uma regra do tipo "contém CONFIRMADA" marcaria `IRREGULARIDADE NAO CONFIRMADA` como confirmada. Qualquer grafia nova, fora do mapa, vira nulo e faz a checagem falhar.

**Resultado esperado:** 1.389 linhas; 871 irregularidades confirmadas (62,7%).

**Pré-requisito:** `silver.unidades_consumidoras` gravada.

## 1. Parâmetros do ambiente

In [0]:
catalog = "v_energy"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"
gold_schema = f"{catalog}.gold"
landing_path = f"/Volumes/{catalog}/bronze/landing"

from pyspark.sql import functions as F, Row
from datetime import datetime

tabela_bronze = f"{bronze_schema}.inspecoes"
tabela_silver = f"{silver_schema}.inspecoes"

def n_br(n):
    return f"{n:,}".replace(",", ".")

def sem_acento(col):
    c = F.translate(F.upper(F.trim(col)), "ÁÀÂÃÉÊÍÓÔÕÚÜÇ", "AAAAEEIOOOUUC")
    return F.regexp_replace(c, r"\s+", " ")

print(f"origem:  {tabela_bronze}")
print(f"destino: {tabela_silver}")

## 2. Funções de checagem de qualidade

Mesmas funções das outras Silvers, gravando em `silver.dq_log`. O log é gravado **sempre**; a Silver só é gravada se **todas** as checagens passarem.

In [0]:
dq_results = []

def dq_check(table_name: str, check_name: str, df, condition):
    """Executa uma checagem de qualidade: conta quantas linhas violam a condição esperada."""
    total = df.count()
    failed = df.filter(~condition).count()
    passed = failed == 0
    dq_results.append(Row(table_name=table_name, check_name=check_name, total_rows=total,
                          failed_rows=failed, passed=passed, checked_at=datetime.now()))
    print(f"[{'PASS' if passed else 'FAIL'}] {table_name} | {check_name} | {failed}/{total} linhas falharam")

def dq_check_unique(table_name: str, check_name: str, df, key_cols: list):
    """Checagem de qualidade específica para unicidade de chave."""
    total = df.count()
    dupes = df.groupBy(*key_cols).count().filter("count > 1").count()
    passed = dupes == 0
    dq_results.append(Row(table_name=table_name, check_name=check_name, total_rows=total,
                          failed_rows=dupes, passed=passed, checked_at=datetime.now()))
    print(f"[{'PASS' if passed else 'FAIL'}] {table_name} | {check_name} | {dupes} chaves duplicadas de {total} linhas")

def dq_check_contagem(table_name: str, check_name: str, df, esperado: int):
    """Checagem de contagem total: passa só se o número de linhas for exatamente o esperado."""
    total = df.count()
    diferenca = abs(total - esperado)
    passed = diferenca == 0
    dq_results.append(Row(table_name=table_name, check_name=check_name, total_rows=total,
                          failed_rows=diferenca, passed=passed, checked_at=datetime.now()))
    print(f"[{'PASS' if passed else 'FAIL'}] {table_name} | {check_name} | {n_br(total)} linhas (esperado {n_br(esperado)})")

## 3. Leitura da Bronze

Conferência: **1.389** linhas (`ago-set` 511, `out-nov` 495, `dez` 383).

In [0]:
bronze = spark.table(tabela_bronze)
n_bronze = bronze.count()
print(f"Linhas na Bronze: {n_br(n_bronze)}")  # esperado: 1.389
display(bronze.groupBy("aba_origem").count().orderBy("aba_origem"))
display(bronze.limit(5))

## 4. Data da inspeção (log I1 e I2)

**I1 · quatro formatos:** ISO (1.256), hífen (46), ponto (44) e barra (43). Cada padrão é convertido com o seu formato, com a convenção do projeto (hífen = mês primeiro; ponto e barra = dia primeiro).
**Evidência:** a convenção converte **100%** das datas; a ordem trocada deixa sem converter 33 de 46 (hífen), 27 de 44 (ponto) e 26 de 43 (barra).

**I2 · data fora do período da aba:** as datas ISO caem todas no período da própria aba (`ago-set`, `out-nov`, `dez`). As **107** que não caem estão **todas na aba `dez`**, com meses de agosto a novembro, e nenhuma ordem de leitura as coloca em dezembro. Interpretação: a aba indica o mês de **lançamento** na planilha; essas são inspeções feitas antes e lançadas em dezembro.
**Decisão:** **manter a data convertida** (é a data da inspeção) e marcar `data_fora_da_aba = true`.

**Conferência:** 0 datas nulas; 107 fora da aba.

In [0]:
# Tratamentos 1 e 2 · data por padrão + flag de data fora do período da aba
formatos = {"9999-99-99": "yyyy-MM-dd", "99-99-9999": "MM-dd-yyyy", "99.99.9999": "dd.MM.yyyy", "99/99/9999": "dd/MM/yyyy"}
padrao_data = F.regexp_replace(F.trim("data"), "[0-9]", "9")
conv = None
for p, fmt in formatos.items():
    val = F.expr(f"to_date(try_to_timestamp(trim(data), '{fmt}'))")
    conv = F.when(padrao_data == p, val) if conv is None else conv.when(padrao_data == p, val)

meses_aba = {"ago-set": [8, 9], "out-nov": [10, 11], "dez": [12]}
dentro = None
for aba, meses in meses_aba.items():
    c = (F.col("aba_origem") == aba) & F.month("data_inspecao").isin(meses)
    dentro = c if dentro is None else (dentro | c)

df_data = (bronze
           .withColumn("data_original", F.col("data"))
           .withColumn("data_inspecao", conv)
           .withColumn("data_fora_da_aba", ~F.coalesce(dentro, F.lit(False))))

# esperado: 0 | 107
print("data_inspecao nula:", n_br(df_data.filter("data_inspecao IS NULL").count()))
print("data fora da aba:  ", n_br(df_data.filter("data_fora_da_aba").count()))
display(df_data.filter("data_fora_da_aba").groupBy("aba_origem", F.month("data_inspecao").alias("mes")).count().orderBy("mes"))

## 5. Resultado da inspeção (log I3)

**Problema:** 8 grafias para o resultado. **Decisão:** mapa explícito, valor a valor (comparação sem acento e em maiúsculas):

| Resultado na Silver | Grafias na Bronze (linhas) |
|---|---|
| `IRREGULARIDADE CONFIRMADA` | `IRREGULARIDADE CONFIRMADA` (777), `Irregularidade Confirmada` (32), `irregularidade confirmada` (27), `CONFIRMADA` (35) |
| `SEM IRREGULARIDADE` | `SEM IRREGULARIDADE` (448), `sem irregularidade` (33), `Sem Irregularidade` (15), `NADA CONSTA` (22) |

**Por que `NADA CONSTA` é "sem irregularidade":** nas 22, o tipo está vazio e a energia recuperada é zero, exatamente como nas `SEM IRREGULARIDADE`. O texto original fica em `resultado_original` só no notebook.

**Flag:** `irregularidade_confirmada` (booleano), para a fila de perdas e para as taxas de acerto.

**Conferência:** 871 confirmadas e 518 sem irregularidade; 0 nulos.

In [0]:
# Tratamento 3 · resultado: mapa explícito (fora do mapa vira nulo e a checagem falha)
mapa_resultado = {
    "IRREGULARIDADE CONFIRMADA": "IRREGULARIDADE CONFIRMADA",
    "CONFIRMADA": "IRREGULARIDADE CONFIRMADA",
    "SEM IRREGULARIDADE": "SEM IRREGULARIDADE",
    "NADA CONSTA": "SEM IRREGULARIDADE",
}
res_pad = sem_acento(F.col("resultado"))
res = None
for origem, destino in mapa_resultado.items():
    res = F.when(res_pad == origem, F.lit(destino)) if res is None else res.when(res_pad == origem, F.lit(destino))

df_res = (df_data
          .withColumn("resultado_original", F.col("resultado"))
          .withColumn("resultado", res)
          .withColumn("irregularidade_confirmada", F.col("resultado") == "IRREGULARIDADE CONFIRMADA"))

# esperado: IRREGULARIDADE CONFIRMADA 871 | SEM IRREGULARIDADE 518 | nulo 0
display(df_res.groupBy("resultado").count())
display(df_res.groupBy("resultado_original", "resultado").count().orderBy("resultado", F.desc("count")))

## 6. Tipo de irregularidade, energia recuperada e inspetor (log I4 e I5)

**I4 · tipo ausente em 2 formas:** nulo real (441) e `-` (77), total de **518**, exatamente as inspeções sem irregularidade. O `-` vira nulo real. Os 5 tipos (`DESVIO DIRETO`, `BY-PASS`, `MEDIDOR ADULTERADO`, `LACRE VIOLADO`, `LIGACAO CLANDESTINA`) não têm variações; `upper(trim)` só garante o padrão.

**I5 · energia recuperada com vírgula:** 125 valores (`1313,70`) misturados com ponto (`838.10`). A vírgula vira ponto e o valor vai para `decimal(12,2)`. Validado: energia positiva em 100% das confirmadas e zero em 100% das sem irregularidade.

**Inspetor:** `upper(trim)` (a planilha manual tinha espaços sobrando, como `" INSP-006"`).

**Conferência:** 518 tipos nulos; 125 vírgulas convertidas; 0 energias nulas.

In [0]:
# Tratamentos 4 e 5 · tipo, energia e inspetor
tipo_limpo = F.upper(F.trim("tipo_irregularidade"))
df_final = (
    df_res
    .withColumn("tipo_irregularidade", F.when(tipo_limpo.isin("-", ""), F.lit(None).cast("string")).otherwise(tipo_limpo))
    .withColumn("kwh_recuperado_original", F.col("kwh_recuperado"))
    .withColumn("kwh_recuperado", F.expr("try_cast(regexp_replace(trim(kwh_recuperado), ',', '.') as decimal(12,2))"))
    .withColumn("inspetor_original", F.col("inspetor"))
    .withColumn("inspetor", F.upper(F.trim("inspetor")))
    .withColumn("id_uc", F.upper(F.trim("id_uc")))
    .withColumn("id_inspecao", F.upper(F.trim("id_inspecao")))
)

# esperado: 518 | 125 | 0
print("tipo nulo:                ", n_br(df_final.filter("tipo_irregularidade IS NULL").count()))
print("kwh com vírgula:          ", n_br(df_final.filter(F.col("kwh_recuperado_original").contains(",")).count()))
print("kwh nulo após conversão:  ", n_br(df_final.filter("kwh_recuperado IS NULL").count()))
print("inspetor com espaços:     ", n_br(df_final.filter(F.col("inspetor_original") != F.trim("inspetor_original")).count()))
display(df_final.groupBy("tipo_irregularidade").count().orderBy(F.desc("count")))

## 7. Seleção final das colunas

As colunas `*_original` ficam de fora. `aba_origem` e `source_file` são mantidas para auditoria.

In [0]:
colunas_finais = [
    "id_inspecao", "id_uc", "data_inspecao", "data_fora_da_aba", "inspetor",
    "resultado", "irregularidade_confirmada", "tipo_irregularidade", "kwh_recuperado",
    "aba_origem", "source_file",
]
df_silver = df_final.select(*colunas_finais)
df_silver.printSchema()

## 8. Checagens de qualidade

| Checagem | Protege |
|---|---|
| Total = 1.389 e `id_inspecao` único | nenhuma linha perdida ou duplicada |
| Colunas obrigatórias não nulas | conversões (I1, I3, I5) |
| `id_uc` no padrão e existente na Silver de UCs | integridade com o cadastro |
| `resultado` com 2 valores e 871 confirmadas | mapa do resultado (I3) |
| Tipo preenchido **se e somente se** confirmada | coerência entre resultado e tipo (I4) |
| Energia positiva se confirmada e zero se não | coerência entre resultado e energia (I5) |
| Data entre ago/2025 e dez/2025; dentro da aba ou marcada | conversão e flag (I1, I2) |
| Inspetor no padrão `INSP-000` | padronização |

A contagem de 1.389 vale para esta extração; com dados novos, o valor esperado muda.

In [0]:
t = "silver.inspecoes"
dq_results.clear()

def ok(cond):
    return F.coalesce(cond, F.lit(False))

dq_check_contagem(t, "total de linhas = 1.389", df_silver, 1_389)
dq_check_unique(t, "id_inspecao único", df_silver, ["id_inspecao"])

for coluna in ["id_inspecao", "id_uc", "data_inspecao", "data_fora_da_aba", "inspetor", "resultado",
               "irregularidade_confirmada", "kwh_recuperado", "aba_origem"]:
    dq_check(t, f"{coluna} não nulo", df_silver, F.col(coluna).isNotNull())

dq_check(t, "id_uc no padrão UC + 6 dígitos", df_silver, ok(F.col("id_uc").rlike("^UC[0-9]{6}$")))
ucs = spark.table(f"{silver_schema}.unidades_consumidoras").select("id_uc").withColumn("existe", F.lit(True))
dq_check(t, "id_uc existe na Silver de UCs", df_silver.join(ucs, "id_uc", "left"), ok(F.col("existe")))

dq_check(t, "resultado entre os 2 valores", df_silver,
         ok(F.col("resultado").isin("IRREGULARIDADE CONFIRMADA", "SEM IRREGULARIDADE")))
dq_check_contagem(t, "irregularidades confirmadas = 871", df_silver.filter("irregularidade_confirmada"), 871)
dq_check(t, "tipo entre os 5 valores ou nulo", df_silver,
         F.col("tipo_irregularidade").isNull() | ok(F.col("tipo_irregularidade").isin(
             "DESVIO DIRETO", "BY-PASS", "MEDIDOR ADULTERADO", "LACRE VIOLADO", "LIGACAO CLANDESTINA")))
dq_check(t, "tipo preenchido se e somente se confirmada", df_silver,
         F.col("irregularidade_confirmada") == F.col("tipo_irregularidade").isNotNull())
dq_check(t, "energia positiva se confirmada e zero se não", df_silver,
         ok(F.when(F.col("irregularidade_confirmada"), F.col("kwh_recuperado") > 0).otherwise(F.col("kwh_recuperado") == 0)))
dq_check(t, "data entre 2025-08-01 e 2025-12-31", df_silver,
         ok(F.col("data_inspecao").between(F.lit("2025-08-01").cast("date"), F.lit("2025-12-31").cast("date"))))
dq_check(t, "data fora da aba só na aba dez (lançamento tardio)", df_silver.filter("data_fora_da_aba"),
         ok(F.col("aba_origem") == "dez"))
dq_check(t, "inspetor no padrão INSP-000", df_silver, ok(F.col("inspetor").rlike("^INSP-[0-9]{3}$")))
dq_check(t, "aba_origem entre as 3 abas", df_silver, ok(F.col("aba_origem").isin("ago-set", "out-nov", "dez")))

## 9. Registro das checagens em `silver.dq_log`

Gravado **sempre**, mesmo com falha (modo `append`).

In [0]:
spark.createDataFrame(dq_results).write.format("delta").mode("append").option("mergeSchema", "true") \
    .saveAsTable(f"{silver_schema}.dq_log")
display(spark.table(f"{silver_schema}.dq_log").filter(F.col("table_name") == t)
        .orderBy(F.col("checked_at").desc()).limit(len(dq_results)))

## 10. Gravação da Silver, só se todas as checagens passarem

In [0]:
falhas = [r for r in dq_results if not r.passed]
if falhas:
    print(f"{len(falhas)} checagem(ns) falharam. {tabela_silver} NÃO foi gravada:")
    for r in falhas:
        print(f"  - {r.check_name}: {n_br(r.failed_rows)} falha(s) em {n_br(r.total_rows)} linhas")
    raise Exception(f"{tabela_silver} não gravada: {len(falhas)} checagem(ns) de qualidade falharam (veja silver.dq_log)")

df_silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela_silver)
print(f"Todas as {len(dq_results)} checagens passaram. {tabela_silver} gravada: {n_br(spark.table(tabela_silver).count())} linhas")

## 11. Bronze × Silver, com as mesmas inspeções

In [0]:
silver_t = spark.table(tabela_silver)
cols_ver = ["id_inspecao", "data", "resultado", "tipo_irregularidade", "kwh_recuperado", "inspetor", "aba_origem"]

def comparar(titulo, ids_df, n=4):
    ids = [r["id_inspecao"] for r in ids_df.select("id_inspecao").limit(n).collect()]
    print(f"=== {titulo} | ids: {ids}")
    print("Bronze (texto original):")
    display(bronze.filter(F.col("id_inspecao").isin(ids)).select(*cols_ver).orderBy("id_inspecao"))
    print("Silver (tratado):")
    display(silver_t.filter(F.col("id_inspecao").isin(ids)).orderBy("id_inspecao"))

comparar("Datas fora do ISO (uma de cada formato)",
         bronze.withColumn("p", F.regexp_replace(F.trim("data"), "[0-9]", "9")).filter("p != '9999-99-99'").dropDuplicates(["p"]))
comparar("Data fora da aba", silver_t.filter("data_fora_da_aba"), n=3)
comparar("Grafias do resultado (uma de cada)", bronze.dropDuplicates(["resultado"]), n=8)
comparar("Energia com vírgula", bronze.filter(F.col("kwh_recuperado").contains(",")), n=3)